In [ ]:
%pip install xarray rioxarray netCDF4 

  Using cached rioxarray-0.18.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached click_plugins-1.1.1-py2.py3-none-any.whl.metadata (6.4 kB)
Using cached rioxarray-0.18.2-py3-none-any.whl (61 kB)
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
    --------------------------------------- 0.1/6.3 MB 4.3 MB/s eta 0:00:02
   --- ------------------------------------ 0.6/6.3 MB 7.1 MB/s eta 0:00:01
   ------ --------------------------------- 1.0/6.3 MB 7.7 MB/s eta 0:00:01
   -------- ------------------------------- 1.4/6.3 MB 7.9 MB/s eta 0:00:01
   ------------ --------------------------- 1.9/6.3 MB 8.6 MB/s eta 0:00:01
   -------------- ------------------------- 2.3/6.3 MB 8.6 MB/s eta 0:00:01
   ------------------ --------------------- 2.9/6.3 MB 9.4 MB/s eta 0:00:01
   -------------


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install numpy pandas fsspec "xarray[io]" requests s3fs aiohttp

  Using cached aiobotocore-2.21.1-py3-none-any.whl.metadata (24 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
     ---------------------------------------- 0.0/71.4 kB ? eta -:--:--
     ---------------------------------------- 71.4/71.4 kB 3.8 MB/s eta 0:00:00
  Using cached aioitertools-0.12.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached botocore-1.37.1-py3-none-any.whl.metadata (5.7 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/443.6 kB ? eta -:--:--
   --------- ------------------------------ 102.4/443.6 kB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 443.6/443.6 kB 5.6 MB/s eta 0:00:00
Using cached aiobotocore-2.21.1-py3-none-any.whl (78 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
from datetime import datetime
import requests

import pandas as pd 
import numpy as np

import fsspec
import xarray as xr
import s3fs  # Use s3fs instead of fsspec


## Subset of Variables

In [ ]:
def get_power_data(url, var):
    # Try direct HTTP access - this is a workaround for S3 access issues
    try:
        # First, let's check if the resource exists
        metadata_url  = f'{url}/.zmetadata'
        response = requests.head(metadata_url)
        print(f"Resource check status: {response.status_code}")
        
        # Use the HTTP URL
        filepath = url
        
        # Open the dataset - try with minimal options
        ds = xr.open_dataset(filepath, engine='zarr')
        
        print("Dataset opened successfully!")
        # print(f"Dataset variables: {list(ds.data_vars)}")
        print(f"Dataset dimensions: {ds.dims}")
        
        ds
        
        
        ds_region = ds[var].sel(
        lat=slice(-38.6, 17.81),
        lon=slice(-93.75, 36.65),
        time=slice("1990-01-01", "2024-12-31")
    ).load()
        print(f"Region data shape: {ds_region.shape}")
        
        # Save to a simple format first
        ds_region.to_netcdf(f"{var}.nc")
        print("Region saved as NetCDF")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Attempting alternative approach...")
        
        try:
            import fsspec
            
            # Try with fsspec without consolidated metadata
            filepath = url
            mapper = fsspec.get_mapper(filepath)
            ds = xr.open_zarr(mapper, consolidated=False)
            
            print("Dataset opened successfully with alternative method!")

            ds_region = ds[var].sel(
        lat=slice(-38.6, 17.81),
        lon=slice(-93.75, 36.65),
        time=slice("1990-01-01", "2024-12-31")
    ).load()
            
            # Save to NetCDF
            ds_region.to_netcdf(f"{var}.nc")
            print("Region saved as NetCDF")
            
        except Exception as e2:
            print(f"Alternative approach also failed: {e2}")

In [26]:
url = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_daily_temporal_lst.zarr'
for var in ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP', 'ALLSKY_SFC_PAR_DIFF', 'ALLSKY_SFC_PAR_DIRH', 'ALLSKY_SFC_PAR_TOT', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_SFC_SW_DIRH', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_UP', 'ALLSKY_SFC_UV_INDEX', 'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SRF_ALB', 'AOD_55', 'AOD_55_ADJ', 'AOD_84', 'CLOUD_AMT', 'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT', 'CLOUD_OD', 'CLRSKY_DAYS', 'CLRSKY_KT', 'CLRSKY_NKT', 'CLRSKY_SFC_LW_DWN', 'CLRSKY_SFC_LW_UP', 'CLRSKY_SFC_PAR_DIFF', 'CLRSKY_SFC_PAR_DIRH', 'CLRSKY_SFC_PAR_TOT', 'CLRSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIRH', 'CLRSKY_SFC_SW_DNI', 'CLRSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_UP', 'CLRSKY_SRF_ALB', 'MIDDAY_INSOL', 'ORIGINAL_ALLSKY_SFC_SW_DIFF', 'ORIGINAL_ALLSKY_SFC_SW_DIRH', 'PSH', 'PW', 'SRF_ALB_ADJ', 'TOA_SW_DNI', 'TOA_SW_DWN', 'TS_ADJ']:    
    get_power_data(url, var)

Resource check status: 200
Dataset opened successfully!
Dataset variables: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP', 'ALLSKY_SFC_PAR_DIFF', 'ALLSKY_SFC_PAR_DIRH', 'ALLSKY_SFC_PAR_TOT', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_SFC_SW_DIRH', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_UP', 'ALLSKY_SFC_UV_INDEX', 'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SRF_ALB', 'AOD_55', 'AOD_55_ADJ', 'AOD_84', 'CLOUD_AMT', 'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT', 'CLOUD_OD', 'CLRSKY_DAYS', 'CLRSKY_KT', 'CLRSKY_NKT', 'CLRSKY_SFC_LW_DWN', 'CLRSKY_SFC_LW_UP', 'CLRSKY_SFC_PAR_DIFF', 'CLRSKY_SFC_PAR_DIRH', 'CLRSKY_SFC_PAR_TOT', 'CLRSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIRH', 'CLRSKY_SFC_SW_DNI', 'CLRSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_UP', 'CLRSKY_SRF_ALB', 'MIDDAY_INSOL', 'ORIGINAL_ALLSKY_SFC_SW_DIFF', 'ORIGINAL_ALLSKY_SFC_SW_DIRH', 'PSH', 'PW', 'SRF_ALB_ADJ', 'TOA_SW_DNI', 'TOA_SW_DWN', 'TS_ADJ']
Dataset dimensions: FrozenMappingWarningOnValuesAccess({'time': 10593, 'lat': 18

## All variables

In [109]:
import requests
import xarray as xr
import fsspec

def get_power_data(url):
    # Try direct HTTP access - this is a workaround for S3 access issues
    try:
        # First, let's check if the resource exists
        metadata_url = f'{url}/.zmetadata'
        response = requests.head(metadata_url)
        print(f"Resource check status: {response.status_code}")
        
        # Use the HTTP URL
        filepath = url
        
        # Open the dataset - try with minimal options
        ds = xr.open_dataset(filepath, engine='zarr')
        
        print("Dataset opened successfully!")
        print(f"Dataset variables: {list(ds.data_vars)}")
        print(f"Dataset dimensions: {ds.dims}")
        
        # Loop through all the variables in the dataset
        for var in ds.data_vars:
            print(f"Processing variable: {var}")
            
            try:
                ds_region = ds[var].sel(
                    lat=slice(-38.6, 17.81),
                    lon=slice(-93.75, 36.65),
                    time=slice("1990-01-01", "2024-12-31")
                ).load()
                
                # Save to a NetCDF file named after the variable
                ds_region.to_netcdf(f"{var}.nc")
                print(f"Region data for {var} saved as {var}.nc")
                
            except Exception as e:
                print(f"Error processing variable {var}: {e}")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Attempting alternative approach...")
        
        try:
            # Try with fsspec without consolidated metadata
            mapper = fsspec.get_mapper(url)
            ds = xr.open_zarr(mapper, consolidated=False)
            
            print("Dataset opened successfully with alternative method!")
            
            # Loop through all the variables in the dataset
            for var in ds.data_vars:
                print(f"Processing variable: {var}")
                
                try:
                    ds_region = ds[var].sel(
                        lat=slice(-38.6, 17.81),
                        lon=slice(-93.75, 36.65),
                        time=slice("1990-01-01", "2024-12-31")
                    ).load()
                    
                    # Save to a NetCDF file named after the variable
                    ds_region.to_netcdf(f"{var}.nc")
                    print(f"Region data for {var} saved as {var}.nc")
                
                except Exception as e:
                    print(f"Error processing variable {var}: {e}")
            
        except Exception as e2:
            print(f"Alternative approach also failed: {e2}")

# Call the function with the URL
url = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_monthly_temporal_lst.zarr'
get_power_data(url)

Resource check status: 200
Dataset opened successfully!
Dataset variables: ['AIRMASS', 'AIRMASS_00', 'AIRMASS_01', 'AIRMASS_02', 'AIRMASS_03', 'AIRMASS_04', 'AIRMASS_05', 'AIRMASS_06', 'AIRMASS_07', 'AIRMASS_08', 'AIRMASS_09', 'AIRMASS_10', 'AIRMASS_11', 'AIRMASS_12', 'AIRMASS_13', 'AIRMASS_14', 'AIRMASS_15', 'AIRMASS_16', 'AIRMASS_17', 'AIRMASS_18', 'AIRMASS_19', 'AIRMASS_20', 'AIRMASS_21', 'AIRMASS_22', 'AIRMASS_23', 'ALLSKY_KT', 'ALLSKY_KT_MAX', 'ALLSKY_KT_MIN', 'ALLSKY_KT_SD', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_DWN_00', 'ALLSKY_SFC_LW_DWN_01', 'ALLSKY_SFC_LW_DWN_02', 'ALLSKY_SFC_LW_DWN_03', 'ALLSKY_SFC_LW_DWN_04', 'ALLSKY_SFC_LW_DWN_05', 'ALLSKY_SFC_LW_DWN_06', 'ALLSKY_SFC_LW_DWN_07', 'ALLSKY_SFC_LW_DWN_08', 'ALLSKY_SFC_LW_DWN_09', 'ALLSKY_SFC_LW_DWN_10', 'ALLSKY_SFC_LW_DWN_11', 'ALLSKY_SFC_LW_DWN_12', 'ALLSKY_SFC_LW_DWN_13', 'ALLSKY_SFC_LW_DWN_14', 'ALLSKY_SFC_LW_DWN_15', 'ALLSKY_SFC_LW_DWN_16', 'ALLSKY_SFC_LW_DWN_17', 'ALLSKY_SFC_LW_DWN_18', 'ALLSKY_SFC_LW_DWN_19',

KeyboardInterrupt: 